# Day 32: LLM Evaluation and Quality Gates
**Golden datasets, prompt regression tests, tool-call fidelity, precision and recall, citations, BLEU and cost-aware quality gates**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas pytest jsonschema tiktoken

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas pytest jsonschema tiktoken
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. The system under test, with versioned configs
v2 lowers k "to save cost". v3 raises k "to improve recall". Both are realistic changes a teammate might make.

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)
## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.
## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.
## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.
## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.
## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.
## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.
## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
%%writefile rag_system.py
import re, numpy as np
from genai_helpers import llm, embed, usage

TEXT = open("knowledge.md").read()
CHUNKS = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", TEXT)) if s.strip()]
IDS = [f"c{i + 1}" for i in range(len(CHUNKS))]
VECS = embed(CHUNKS)
BASE = "Answer using ONLY the context. If the answer is not there, say \"I don't know\"."
CONFIGS = {
    "v1": {"k": 3, "system": BASE},
    "v2": {"k": 1, "system": "Answer briefly. " + BASE},          # "save cost"
    "v3": {"k": 8, "system": BASE},                               # "improve recall"
}

def retrieve(question, k):
    order = np.argsort(VECS @ embed(question)[0])[::-1][:k]
    return [IDS[i] for i in order], [CHUNKS[i] for i in order]

def answer(question, version="v1", cite=False):
    cfg = CONFIGS[version]
    ids, ctx = retrieve(question, cfg["k"])
    context = "\n".join(f"[{i}] {c}" if cite else c for i, c in zip(ids, ctx))
    out = llm("Context:\n" + context + f"\n\nQuestion: {question}", system=cfg["system"], temperature=0)
    if cite and "[" not in out:                         # offline stand-in: cite the chunk the answer came from
        src = next((i for i, c in zip(ids, ctx) if c.strip(" ") in out or out.strip() in c), None)
        out = out + (f" [{src}]" if src else "")
    return out, ids

def answer_with_usage(question, version="v1"):
    out, ids = answer(question, version)
    return out, ids, usage()

## 3. A golden dataset (versioned JSONL)

In [ ]:
import json, re, importlib, rag_system
importlib.reload(rag_system)
from rag_system import CHUNKS, IDS, answer, retrieve

def cid(fragment):
    return [IDS[i] for i, c in enumerate(CHUNKS) if fragment in c]

GOLDEN = [
    ("leave-001", "How many paid leave days do full-time staff get?", "18 days", ["18 days"], ["8 days"], "18 days of paid leave", "leave"),
    ("leave-002", "How many sick days do I get each year?", "8 days", ["8 days"], ["18 days"], "8 days of paid sick leave", "leave"),
    ("leave-003", "When does carried over leave expire?", "31 March", ["31 March"], ["30 June"], "expires on 31 March", "leave"),
    ("remote-001", "How many days a week can I work remotely?", "2 days", ["2 days"], ["5 days"], "up to 2 days per week", "remote"),
    ("remote-002", "How far ahead must I request remote work abroad?", "14 days", ["14 days"], [], "at least 14 days in advance", "remote"),
    ("travel-001", "Hotel limit per night in Japan?", "15,000 JPY", ["15,000 JPY"], ["6,000 INR per night in Japan"], "15,000 JPY", "travel"),
    ("travel-002", "Deadline for submitting travel expenses?", "30 days", ["30 days"], [], "within 30 days", "travel"),
    ("travel-003", "How early should flights be booked?", "10 days", ["10 days"], [], "10 days before travel", "travel"),
    ("it-001", "What does error VPN-417 mean?", "VPN certificate has expired", ["certificate"], [], "VPN-417", "it"),
    ("it-002", "Minimum password length?", "14 characters", ["14 characters"], [], "14 characters", "it"),
    ("it-003", "How long does a new laptop take?", "5 working days", ["5 working days"], [], "5 working days", "it"),
    ("sec-001", "How fast must I report a security incident?", "1 hour", ["1 hour"], ["24 hours"], "within 1 hour", "security"),
]
with open("golden_v1.jsonl", "w") as f:
    for gid, q, exp, inc, exc, frag, topic in GOLDEN:
        f.write(json.dumps({"id": gid, "question": q, "expected": exp, "must_include": inc, "must_not_include": exc,
                            "relevant_ids": cid(frag), "tags": {"topic": topic, "lang": "en"}, "version": 1}) + "\n")
golden = [json.loads(l) for l in open("golden_v1.jsonl")]
print(len(golden), "golden items. Example:\n", json.dumps(golden[2], indent=2))

## 4. Prompt regression tests with PyTest
The version under test comes from an environment variable, exactly as a CI pipeline would set it.

In [ ]:
%%writefile test_golden.py
import json, os, pytest
from rag_system import answer

VERSION = os.getenv("PROMPT_VERSION", "v1")
GOLDEN = [json.loads(l) for l in open("golden_v1.jsonl")]

@pytest.mark.parametrize("item", GOLDEN, ids=lambda i: i["id"])
def test_golden_item(item):
    out, _ = answer(item["question"], VERSION)
    for phrase in item["must_include"]:
        assert phrase.lower() in out.lower(), f"missing '{phrase}' in: {out}"
    for phrase in item["must_not_include"]:
        assert phrase.lower() not in out.lower(), f"forbidden '{phrase}' in: {out}"

In [ ]:
# ===== Option A: Google Colab (active) =====
!PROMPT_VERSION=v1 python -m pytest -q --color=no -p no:cacheprovider test_golden.py 2>&1 | tail -2
!PROMPT_VERSION=v2 python -m pytest -q --color=no -p no:cacheprovider test_golden.py 2>&1 | tail -2
# ===== Option B: VS Code terminal =====
# Windows PowerShell:  $env:PROMPT_VERSION="v2"; pytest -q test_golden.py
# macOS/Linux:          PROMPT_VERSION=v2 pytest -q test_golden.py

### Per-item comparison: what exactly regressed?

In [ ]:
import pandas as pd

def passes(item, version):
    out, ids = answer(item["question"], version)
    ok = all(p.lower() in out.lower() for p in item["must_include"]) and not any(p.lower() in out.lower() for p in item["must_not_include"])
    return ok, ids

rows = []
for g in golden:
    ok1, _ = passes(g, "v1"); ok2, _ = passes(g, "v2")
    rows.append({"id": g["id"], "v1": ok1, "v2": ok2, "status": "REGRESSED" if ok1 and not ok2 else ("fixed" if ok2 and not ok1 else "")})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))
print(f"\nv1 pass rate {cmp.v1.mean():.2f} | v2 pass rate {cmp.v2.mean():.2f} | regressed: {list(cmp[cmp.status == 'REGRESSED'].id)}")
if not ONLINE:
    print("Offline, the stand-in answers by picking one sentence, so absolute pass rates are low. The comparison between versions is still meaningful.")

## 5. NEW (v4): Tool-call fidelity tests
For agents, the tool call is the answer. We test the tool name, schema-valid arguments, exact values, and that no tool is called when none is needed. Offline, a small rule-based chooser stands in for the model's function calling.

In [ ]:
%%writefile tools_under_test.py
import json, os, re

SCHEMAS = {
    "issue_refund": {"type": "object", "required": ["order_id", "amount"], "additionalProperties": False,
                     "properties": {"order_id": {"type": "string", "pattern": "^ORD-[0-9]+$"}, "amount": {"type": "integer", "minimum": 1}}},
    "change_plan": {"type": "object", "required": ["customer_id", "plan"], "additionalProperties": False,
                    "properties": {"customer_id": {"type": "string"}, "plan": {"type": "string", "enum": ["Basic", "Family 5G", "Unlimited"]}}},
    "lookup_ticket": {"type": "object", "required": ["ticket_id"], "additionalProperties": False,
                      "properties": {"ticket_id": {"type": "string", "pattern": "^T-[0-9]+$"}}},
}

def choose_tool(message):
    """Return (tool_name, args) or (None, None). Offline rule-based stand-in for model function calling.
    Online, send SCHEMAS as tools to the chat completions API and read message.tool_calls instead."""
    m = message.lower()
    order = re.search(r"ORD-\d+", message)
    if "refund" in m and order:
        amount = re.search(r"refund (?:of )?([\d,]+)", m)
        return "issue_refund", {"order_id": order.group(0), "amount": int(amount.group(1).split(",")[0]) if amount else 0}
    plan = next((p for p in ["Basic", "Family 5G", "Unlimited"] if p.lower() in m), None)
    cust = re.search(r"C-\d+", message)
    if ("switch" in m or "change" in m) and plan and cust:
        return "change_plan", {"customer_id": cust.group(0), "plan": plan}
    ticket = re.search(r"T-\d+", message)
    if ticket and ("status" in m or "ticket" in m):
        return "lookup_ticket", {"ticket_id": ticket.group(0)}
    return None, None

In [ ]:
%%writefile test_tool_calls.py
import pytest
from jsonschema import validate
from tools_under_test import SCHEMAS, choose_tool

CASES = [
    ("Please refund 300 on order ORD-77", "issue_refund", {"order_id": "ORD-77", "amount": 300}),
    ("Refund of 50 for ORD-12 please", "issue_refund", {"order_id": "ORD-12", "amount": 50}),
    ("Refund 1,200 on ORD-9", "issue_refund", {"order_id": "ORD-9", "amount": 1200}),
    ("Switch C-1002 to the Family 5G plan", "change_plan", {"customer_id": "C-1002", "plan": "Family 5G"}),
    ("What is the status of ticket T-881?", "lookup_ticket", {"ticket_id": "T-881"}),
    ("Hi, thanks for your help!", None, None),
    ("What is the weather in Osaka?", None, None),
]

@pytest.mark.parametrize("msg,tool,args", CASES, ids=[c[0][:24] for c in CASES])
def test_tool_call(msg, tool, args):
    name, got = choose_tool(msg)
    assert name == tool, f"expected tool {tool}, got {name}"
    if tool:
        validate(got, SCHEMAS[tool])          # schema-valid arguments
        assert got == args, f"args {got} != {args}"

In [ ]:
# ===== Option A: Google Colab (active) =====
!python -m pytest -q --color=no -p no:cacheprovider test_tool_calls.py 2>&1 | tail -4
# ===== Option B: VS Code terminal =====
# pytest -q test_tool_calls.py

One case fails on purpose: the chooser reads `1,200` as `1`. A wrong amount is worse than no call. **TODO:** fix the parsing in `tools_under_test.py` and rerun until all cases pass.

## 6. Context precision, recall and average precision

In [ ]:
def precision(ret, rel):
    return sum(c in rel for c in ret) / len(ret)

def recall(ret, rel):
    return len(set(ret) & set(rel)) / len(rel)

def avg_precision(ret, rel):
    hits, total = 0, 0.0
    for k, c in enumerate(ret, 1):
        if c in rel:
            hits += 1
            total += hits / k
    return total / len(rel)

table = []
for k in [1, 3, 5]:
    p, r, ap = [], [], []
    for g in golden:
        ret, _ = retrieve(g["question"], k)
        rel = set(g["relevant_ids"])
        p.append(precision(ret, rel)); r.append(recall(ret, rel)); ap.append(avg_precision(ret, rel))
    table.append({"k": k, "precision": round(sum(p) / len(p), 2), "recall": round(sum(r) / len(r), 2), "avg_precision": round(sum(ap) / len(ap), 2)})
print(pd.DataFrame(table).to_string(index=False))

## 7. Citations, written eval-first
**NEW (v4): eval-first development with Codex.** First we write the eval for the new behaviour (every answer cites a retrieved, relevant chunk) and watch it fail. Then we turn the feature on (the change you would ask Codex for) and rerun the same eval.

In [ ]:
CITE = re.compile(r"\[(c\d+)\]")

def citation_eval(cite):
    rows = []
    for g in golden:
        out, ids = answer(g["question"], "v1", cite=cite)
        cited = CITE.findall(out)
        rows.append({"id": g["id"], "has_citation": bool(cited),
                     "cited_retrieved": bool(cited) and all(c in ids for c in cited),
                     "cited_relevant": bool(cited) and all(c in g["relevant_ids"] for c in cited)})
    return pd.DataFrame(rows)

before = citation_eval(cite=False)
print("BEFORE (eval written, feature not built):", before[["has_citation", "cited_retrieved", "cited_relevant"]].mean().round(2).to_dict())
after = citation_eval(cite=True)
print("AFTER  (feature built, same eval):        ", after[["has_citation", "cited_retrieved", "cited_relevant"]].mean().round(2).to_dict())

A Codex prompt for the change: *"Make `answer()` include the chunk id in square brackets for every fact it uses. Do not change retrieval. Run `python -m pytest test_golden.py` and the citation eval, and show the diff."* Put the eval command in `AGENTS.md` so Codex runs it after every change.

## 8. BLEU from scratch, and where it misleads

In [ ]:
import math
from collections import Counter

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def bleu(candidate, reference, max_n=4):
    c, r = re.findall(r"\w+", candidate.lower()), re.findall(r"\w+", reference.lower())
    logs = []
    for n in range(1, max_n + 1):
        cn, rn = ngrams(c, n), ngrams(r, n)
        overlap = sum(min(cnt, rn[g]) for g, cnt in cn.items())
        logs.append(math.log((overlap + 1) / (max(1, sum(cn.values())) + 1)))      # add-1 smoothing
    bp = 1.0 if len(c) > len(r) else math.exp(1 - len(r) / max(1, len(c)))
    return round(bp * math.exp(sum(logs) / max_n), 2)

ref = "Carried forward leave expires on 31 March of the following year."
cases = {
    "exact copy": ref,
    "correct paraphrase": "Any leave you roll over must be used by the end of next March.",
    "wrong, similar words": "Carried forward leave expires on 30 June of the following year.",
}
for name, cand in cases.items():
    print(f"{name:<22} BLEU {bleu(cand, ref)}")
print("\nThe wrong answer scores far higher than the correct paraphrase. Use BLEU for translation-like tasks, not open answers.")

## 9. NEW (v4): Token and cost metrics
Prices are **placeholders**, not real prices. Offline token counts are estimates.

In [ ]:
from rag_system import answer_with_usage
PRICE_PER_1M = {"input": 1.00, "output": 4.00}       # PLACEHOLDER prices

def cost_profile(version):
    tin, tout = [], []
    for g in golden:
        _, _, u = answer_with_usage(g["question"], version)
        tin.append(u["input_tokens"]); tout.append(u["output_tokens"])
    cpq = (sum(tin) * PRICE_PER_1M["input"] + sum(tout) * PRICE_PER_1M["output"]) / 1_000_000 / len(golden)
    return {"version": version, "avg_input_tokens": round(sum(tin) / len(tin)), "avg_output_tokens": round(sum(tout) / len(tout)),
            "cost_per_query": round(cpq, 6), "cost_per_1000": round(cpq * 1000, 3)}

print(pd.DataFrame([cost_profile(v) for v in ["v1", "v2", "v3"]]).to_string(index=False))
print("(placeholder prices)")

## 10. A cost-aware quality gate for CI

In [ ]:
%%writefile quality_gate.py
import json, os, sys
from rag_system import answer_with_usage

VERSION = sys.argv[1] if len(sys.argv) > 1 else "v1"
THRESHOLDS = {"recall": 0.85}          # absolute floor for retrieval
MAX_DROP = 0.05                         # allowed pass-rate drop versus the baseline (v1)
MAX_COST_RISE = 0.20                    # allowed rise in cost per query versus the baseline
PRICE_PER_1M = {"input": 1.00, "output": 4.00}   # PLACEHOLDER prices

golden = [json.loads(l) for l in open("golden_v1.jsonl")]
passed, recalls, tin, tout = 0, [], 0, 0
for g in golden:
    out, ids, u = answer_with_usage(g["question"], VERSION)
    ok = all(p.lower() in out.lower() for p in g["must_include"]) and not any(p.lower() in out.lower() for p in g["must_not_include"])
    passed += ok
    recalls.append(len(set(ids) & set(g["relevant_ids"])) / len(g["relevant_ids"]))
    tin += u["input_tokens"]; tout += u["output_tokens"]
cost = (tin * PRICE_PER_1M["input"] + tout * PRICE_PER_1M["output"]) / 1_000_000 / len(golden)
scores = {"pass_rate": round(passed / len(golden), 2), "recall": round(sum(recalls) / len(recalls), 2),
          "avg_input_tokens": round(tin / len(golden)), "cost_per_query": round(cost, 6)}
failures = [f"{k} {scores[k]} < {v}" for k, v in THRESHOLDS.items() if scores[k] < v]
if VERSION != "v1" and os.path.exists("report_v1.json"):
    base = json.load(open("report_v1.json"))["scores"]
    if scores["pass_rate"] < base["pass_rate"] - MAX_DROP:
        failures.append(f"pass_rate dropped {base['pass_rate'] - scores['pass_rate']:.2f} vs baseline")
    rise = scores["cost_per_query"] / base["cost_per_query"] - 1
    if rise > MAX_COST_RISE:
        failures.append(f"cost per query up {rise:.0%} vs baseline (placeholder prices)")
print(f"{VERSION}: {'FAIL' if failures else 'PASS'} {scores}" + (f" -> {failures}" if failures else ""))
json.dump({"version": VERSION, "scores": scores, "failures": failures}, open(f"report_{VERSION}.json", "w"), indent=2)
sys.exit(1 if failures else 0)

In [ ]:
# ===== Option A: Google Colab (active) =====
!python quality_gate.py v1 && echo "Gate passed: release allowed"
!python quality_gate.py v2 || echo "Gate failed: release blocked"
!python quality_gate.py v3 || echo "Gate failed: release blocked"
# ===== Option B: VS Code terminal / Azure Pipelines step =====
# python quality_gate.py v3      (a non-zero exit code fails the pipeline stage)

v2 fails on quality (recall). v3 retrieves more but every query now carries much more context: it fails on cost. A change like v3 needs a human decision about whether the quality gain is worth the price.

### TODO: with a real model (API key), add an absolute threshold of `pass_rate >= 0.9` to `THRESHOLDS`, add an `avg_output_tokens` limit, and rerun the gate for all three versions.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)